# Part 2: Bayesian Probability on IMDb Movie Reviews
**Formative 3 - Probability Distributions, Bayesian Probability, and Gradient Descent Implementation**

**Question:** if a review contains a given keyword, how likely is it to be positive?

**Our choice:** we compute **P(Positive | keyword)** only, as the assignment requires (not both directions).

**Method:** Bayes' theorem implemented with basic Python only (`csv` and `re`). A keyword is counted once per review.

$$P(\text{Positive}\mid kw)=\frac{P(kw\mid \text{Positive})\;P(\text{Positive})}{P(kw)}$$

| Term | Meaning |
|---|---|
| Prior `P(Positive)` | share of all reviews that are positive, before looking at any word |
| Likelihood `P(kw\|Positive)` | among positive reviews, the share containing the keyword |
| Marginal `P(kw)` | among all reviews, the share containing the keyword |
| Posterior `P(Positive\|kw)` | among reviews containing the keyword, the share that are positive |

## 1. Setup (works locally and on Colab)

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/igorntwari/ml-formative-3-probability-bayes-gradient-descent.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not os.path.exists("bayes_imdb.py"):
    subprocess.run(["git", "clone", REPO], check=True)
    os.chdir("ml-formative-3-probability-bayes-gradient-descent/part2_bayesian")

CSV = "IMDB Dataset.csv"
if not os.path.exists(CSV):
    if IN_COLAB:
        from google.colab import files
        print("Upload IMDB Dataset.csv")
        files.upload()
    else:
        raise FileNotFoundError("Put 'IMDB Dataset.csv' next to this notebook (part2_bayesian/).")

print("Working dir:", os.getcwd())

## 2. Load the data

In [ ]:
from bayes_imdb import *

reviews = load_reviews(CSV)
n_total = len(reviews)
n_pos = sum(1 for _, label in reviews if label == "positive")
print(f"Reviews: {n_total} | positive: {n_pos} | negative: {n_total - n_pos}")
print(f"Prior P(Positive) = {n_pos / n_total:.4f}")

The dataset is balanced, so the prior is 0.5. Any keyword whose posterior moves away from 0.5 therefore carries real information.

## 3. Keywords and justification
*(Edit the one-line reasons to match your final lists in `bayes_imdb.py`.)*

- **Positive keywords:** *reason for each, e.g. direct praise, frequent, clear meaning on its own*
- **Negative keywords:** *reason for each, e.g. direct criticism, frequent, clear meaning on its own*
- **How we chose them:** we ranked words by their positive share and kept only words that appear in many reviews, so the estimates are stable.

In [ ]:
print("Positive:", POSITIVE_KEYWORDS)
print("Negative:", NEGATIVE_KEYWORDS)

## 4. Probability table (prior, likelihood, marginal, posterior)

In [ ]:
keywords = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS
results = [analyze_keyword(reviews, k, n_total, n_pos) for k in keywords]
print_table(results)

## 5. Verification: Bayes result vs direct count
The posterior must equal (positive reviews with the word) / (all reviews with the word).

In [ ]:
print(f"{'keyword':<12}{'bayes':>10}{'direct':>10}  match")
for r in results:
    direct = direct_posterior(reviews, r["keyword"])
    ok = abs(r["posterior"] - direct) < 1e-9
    print(f"{r['keyword']:<12}{r['posterior']:>10.4f}{direct:>10.4f}  {ok}")
    assert ok, f"Mismatch for {r['keyword']}"
print("All keywords verified.")

## 6. Chart: posterior vs the 0.5 starting rate

In [ ]:
import matplotlib.pyplot as plt

words = [r["keyword"] for r in results]
posts = [r["posterior"] for r in results]
colors = ["tab:green" if w in POSITIVE_KEYWORDS else "tab:red" for w in words]

plt.figure(figsize=(9, 4.5))
plt.bar(words, posts, color=colors)
plt.axhline(0.5, color="black", linestyle="--", label="Prior P(Positive) = 0.5")
plt.ylabel("P(Positive | keyword)")
plt.title("How much each keyword shifts the probability of a positive review")
plt.ylim(0, 1)
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("posterior_chart.png", dpi=150)
plt.show()

## 7. Interpretation

In [ ]:
for r in results:
    shift = (r["posterior"] - r["prior"]) * 100
    direction = "above" if shift > 0 else "below"
    print(f"- '{r['keyword']}' appears in {r['marginal']:.1%} of reviews. "
          f"{r['posterior']:.1%} of those are positive, {abs(shift):.1f} points {direction} the 50% baseline.")

**Reading the results** *(rewrite in your own words before presenting)*

- Positive keywords push the probability well above 0.5, negative keywords well below it.
- The marginal `P(kw)` is in the denominator because we condition on reviews that contain the word, so we divide by how common the word is overall.
- A common word that appears equally in both classes (for example "good", about 0.49) gives almost no information, even though its likelihood is high.

**Limitations**
- Negation ("not good") and sarcasm are not handled, because we look at one word at a time.
- Rare words give unstable estimates, so we chose words that appear in thousands of reviews.
- Keywords were chosen using the same dataset they are evaluated on, so the numbers describe this dataset, not all movie reviews.

## 8. Save the results

In [ ]:
import csv
with open("part2_results.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["keyword", "prior", "likelihood", "marginal", "posterior"])
    w.writeheader()
    w.writerows(results)
print("Saved part2_results.csv and posterior_chart.png")